# Load Dependencies

In [ ]:
# Data analysis libraries
import numpy as np
import pandas as pd; pd.options.display.max_columns = 200
import geopandas as gpd
import linref as lr

# Visualization libraries
import plotly.express as px

# Utility libraries
import os

In [ ]:
# Define global variables
PROJECT_CRS = 'EPSG:3857'

# Load Data

In [ ]:
# Point this to the location of the Geopackage file
fp = os.path.join('..', '99_Resources', 'franklin_county_training_data.gpkg')

# List all the layers in the file
gpd.list_layers(fp)

In [ ]:
# Load roadway data
roadways = gpd.read_file(fp, layer='roadways')
roadways.to_crs(PROJECT_CRS, inplace=True)

# Load crash data
query = """
SELECT OBJECTID, CRASH_YR, KABCO, CRASH_TYPE_SIMPLE, DAY_IN_WEEK_TEXT, HOUR_PERIOD, geom
FROM crashes_enriched
"""
crashes = gpd.read_file(fp, sql=query)
crashes.to_crs(PROJECT_CRS, inplace=True)

print(f'Data loaded: {len(roadways):,.0f} roadways, {len(crashes):,.0f} crashes')

# Crash Scoring Metrics

In [ ]:
# Create a series of crash scoring metrics based on crash severity and mode
# These will be used for creating a variety of high-injury networks for each defined
# scoring metric

# We will first create boolean masks for each of the scoring metrics

# Crash severity metrics
mask_kabc = crashes['KABCO'].isin(['K', 'A', 'B', 'C'])
mask_ka   = crashes['KABCO'].isin(['K', 'A'])

# Crash mode metrics
mask_ped = crashes['CRASH_TYPE_SIMPLE'].isin(['Pedestrian'])
mask_pdc = crashes['CRASH_TYPE_SIMPLE'].isin(['Pedalcycle'])
mask_veh = ~(mask_ped | mask_pdc)

In [ ]:
# Combined metrics
crashes['SCORE_KABC_VEH'] = mask_kabc * mask_veh * 1
crashes['SCORE_KABC_PED'] = mask_kabc * mask_ped * 1
crashes['SCORE_KABC_PDC'] = mask_kabc * mask_pdc * 1
crashes['SCORE_KA_VEH']   = mask_ka   * mask_veh * 1
crashes['SCORE_KA_PED']   = mask_ka   * mask_ped * 1
crashes['SCORE_KA_PDC']   = mask_ka   * mask_pdc * 1

In [ ]:
# How can we create metrics for late night weekend crashes?
# E.g., Friday and Saturday nights between 9 PM and 3 AM

In [ ]:
crashes['SCORE_KA_PED'].sum()

In [ ]:
crashes.head()

# Linear Referencing

In [ ]:
# Create events collections for managing linearly referenced data
roadways_ec = lr.EventsCollection(roadways, keys=['NLF_ID'], beg='CTL_BEGIN_', end='CTL_END_NB', geom='geometry')
crashes_ec = lr.EventsCollection(crashes, keys=['NLFID'], beg='COUNTY_LOG_NBR', geom='geometry')

In [ ]:
get_columns = ['SPEED_LIMI', 'LANES_NBR', 'LANE_WIDTH']
crashes_ec.df[get_columns] = crashes_ec.merge(roadways_ec)[get_columns].most()

# Crash Distribution Analysis

_Linref-based Crash Distribution Schematic Example_

<img src="../99_Resources/img/linref-distribution-schematic.png" width=1000>